In [8]:
import json
import copy
import re

INPUT_FILE = "input.qsf"
OUTPUT_FILE = "generated.qsf"
TOTAL_QUESTIONS = 60

with open(INPUT_FILE) as f:
    data = json.load(f)

elements = data["SurveyElements"]

# -------------------------
# find template question
# -------------------------

template = None

for el in elements:
    if el["Element"] == "SQ":
        tag = el["Payload"].get("DataExportTag", "")
        if tag == "R1.1":
            template = el
            break

if template is None:
    raise Exception("Could not find question with DataExportTag R1.1")

# -------------------------
# find max existing QID
# -------------------------

qid_numbers = []

for el in elements:
    if el["Element"] == "SQ":
        qid = el["PrimaryAttribute"]
        num = int(re.sub(r"\D", "", qid))
        qid_numbers.append(num)

max_qid = max(qid_numbers)

# -------------------------
# locate block
# -------------------------

block = None

for el in elements:
    if el["Element"] == "BL":
        block = el

standard_block = None

for key in block["Payload"]:
    if block["Payload"][key]["Type"] != "Trash":
        standard_block = block["Payload"][key]

# -------------------------
# generate questions
# -------------------------

new_questions = []

for i in range(2, TOTAL_QUESTIONS + 1):

    new_qid_num = max_qid + (i - 1)
    new_qid = f"QID{new_qid_num}"

    q = copy.deepcopy(template)

    q["PrimaryAttribute"] = new_qid
    q["SecondaryAttribute"] = f"Question-{i}"

    payload = q["Payload"]

    payload["QuestionID"] = new_qid
    payload["QuestionText"] = f"Question-{i}"
    payload["QuestionDescription"] = f"Question-{i}"
    payload["DataExportTag"] = f"R1.{i}"

    payload["Choices"]["1"]["Display"] = f"R1.{i}-RESPONSE"
    payload["Choices"]["2"]["Display"] = f"R1.{i}-REACTION-TIME"
    payload["Choices"]["3"]["Display"] = f"R1.{i}-TYPING-TIME"

    new_questions.append(q)

    standard_block["BlockElements"].append({
        "Type": "Question",
        "QuestionID": new_qid
    })

# -------------------------
# update QC
# -------------------------

for el in elements:
    if el["Element"] == "QC":
        el["SecondaryAttribute"] = str(TOTAL_QUESTIONS)

# -------------------------
# append questions
# -------------------------

data["SurveyElements"].extend(new_questions)

# -------------------------
# save
# -------------------------

with open(OUTPUT_FILE, "w") as f:
    json.dump(data, f, indent=2)

print("Generated:", OUTPUT_FILE)

Generated: generated.qsf
